In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

In [ ]:
def get_spark(app_name: str) -> SparkSession:
    master = os.getenv("SPARK_MASTER")  # ex: spark://spark-master-rss:7077
    builder = SparkSession.builder.appName(app_name)
    if master:
        builder = builder.master(master)
    return builder.getOrCreate()

def main():
    spark = get_spark("P02-Treino")

    data_path = os.getenv("DATA_PATH", "/opt/spark/dados/dataset.csv")
    model_path = os.getenv("MODEL_PATH", "/opt/spark/dados/modelo")
    pred_path  = os.getenv("PRED_PATH", "/opt/spark/dados/previsoesteste")

    df = spark.read.csv(data_path, header=True, inferSchema=True)
    df = df.withColumn("Date", col("Date").cast("date"))
    df = df.withColumn("Year", year(col("Date"))).withColumn("Month", month(col("Date")))

    # ✅ Split temporal (80% primeiras datas treino, 20% últimas datas teste)
    df = df.orderBy("Date")
    n = df.count()
    cutoff = int(n * 0.8)

    df_idx = df.rdd.zipWithIndex().toDF(["row", "idx"]).select("row.*", "idx")
    train = df_idx.filter(col("idx") < cutoff).drop("idx")
    test  = df_idx.filter(col("idx") >= cutoff).drop("idx")

    assembler = VectorAssembler(inputCols=["Year", "Month"], outputCol="Features")
    lr = LinearRegression(featuresCol="Features", labelCol="Sales")
    pipeline = Pipeline(stages=[assembler, lr])

    model = pipeline.fit(train)
    pred = model.transform(test)

    rmse = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="rmse").evaluate(pred)
    r2   = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="r2").evaluate(pred)

    print(f"RMSE = {rmse}")
    print(f"R2   = {r2}")

    model.write().overwrite().save(model_path)
    pred.select("Date", "Sales", "prediction").write.mode("overwrite").option("header", True).csv(pred_path)

    spark.stop()

if __name__ == "__main__":
    main()    